# Phase 10.3 — Normalization & Ranking Dataset

Normalize raw `RankingFeatures` within a query group and build dataset rows.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.benchmark_labels import binary_relevance_labels_for_products
from productiq.ranking.dataset_builder import RankingDatasetQueryGroup, build_ranking_dataset_from_query_groups
from productiq.ranking.feature_schema import RankingFeatures
from productiq.ranking.normalization import normalize_feature_rows
from productiq.retrieval.evaluation.dataset import LexicalEvaluationQuery

In [ ]:
rows = (
    RankingFeatures(
        product_id="P1",
        retrieval={"bm25_score": 1.0, "retrieved_by_bm25": True, "retrieved_by_vector": False, "retrieved_by_both": False},
        matching={"matched_attribute_count": 0},
        catalog={"discount_price_inr": 500, "original_price_inr": 999, "discount_amount_inr": 499},
    ),
    RankingFeatures(
        product_id="P2",
        retrieval={"bm25_score": 3.0, "retrieved_by_bm25": True, "retrieved_by_vector": False, "retrieved_by_both": False},
        matching={"matched_attribute_count": 0},
        catalog={"discount_price_inr": 700, "original_price_inr": 1200, "discount_amount_inr": 500},
    ),
)
normalized = normalize_feature_rows(rows)
eval_query = LexicalEvaluationQuery(
    query_id="q_demo",
    query_text="demo",
    category="demo",
    relevant_product_ids=("P1",),
)
labels = binary_relevance_labels_for_products(eval_query, [row.product_id for row in rows])
dataset = build_ranking_dataset_from_query_groups(
    (
        RankingDatasetQueryGroup(
            query_id=eval_query.query_id,
            features=rows,
            relevance_label_by_product_id=labels,
        ),
    ),
    query_set_version="demo",
    judgment_source_version="manual",
)
dataset.model_dump(mode="json")